# Credit Risk Assessment with Custom Feature Engineering

## IntegratedML Pluggable Models Demo 1

This notebook demonstrates how to implement and use custom feature engineering within IntegratedML workflows for credit risk assessment. We'll showcase:

1. **Data Generation & Exploration** - Creating realistic synthetic credit data
2. **Custom Feature Engineering** - Domain-specific financial transformations
3. **Model Training & Evaluation** - Custom classifier with multiple configurations
4. **Business Impact Analysis** - Interpreting results for business decisions
5. **IntegratedML Integration** - Database-resident ML workflows

### Key Learning Objectives
- How to implement custom feature engineering in IntegratedML
- Security benefits of processing sensitive data without export
- Performance advantages of database-resident processing
- scikit-learn compatibility with enterprise enhancements

## Setup and Imports

In [ ]:
# Standard library imports
import sys
import os
import warnings
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parents[2]
sys.path.append(str(project_root))

# Data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
import joblib

# Project imports
from demos.credit_risk.models.credit_risk_classifier import CustomCreditRiskClassifier
from demos.credit_risk.data.generate_sample_data import CreditDataGenerator
from demos.credit_risk.scripts.data_preprocessing import CreditDataPreprocessor, create_feature_summary

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')

print("✅ Setup complete!")
print(f"📁 Project root: {project_root}")

## 1. Data Generation and Exploration

Let's start by generating realistic synthetic credit application data that mimics real-world patterns.

In [ ]:
# Generate synthetic credit risk data
print("🏗️ Generating synthetic credit application data...")

generator = CreditDataGenerator(random_seed=42)
X, y = generator.generate_dataset(n_samples=1000, default_rate=0.30)

print(f"📊 Generated dataset: {X.shape[0]} applications, {X.shape[1]} features")
print(f"📈 Default rate: {y.mean():.1%}")
print(f"💰 Average credit amount: ${X['credit_amount'].mean():,.0f}")
print(f"👥 Age range: {X['age'].min()}-{X['age'].max()} years")

# Display first few rows
display(X.head())

In [ ]:
# Explore data distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Credit Application Data Distributions', fontsize=16)

# Age distribution
axes[0,0].hist(X['age'], bins=20, alpha=0.7, color='skyblue')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')

# Credit amount distribution
axes[0,1].hist(X['credit_amount'], bins=30, alpha=0.7, color='lightgreen')
axes[0,1].set_title('Credit Amount Distribution')
axes[0,1].set_xlabel('Credit Amount ($)')

# Employment duration
axes[0,2].hist(X['employment_duration'], bins=25, alpha=0.7, color='orange')
axes[0,2].set_title('Employment Duration')
axes[0,2].set_xlabel('Duration (months)')

# Default rate by age group
age_groups = pd.cut(X['age'], bins=[0, 25, 35, 45, 55, 100], labels=['<25', '25-34', '35-44', '45-54', '55+'])
default_by_age = pd.DataFrame({'age_group': age_groups, 'default': y}).groupby('age_group')['default'].mean()
axes[1,0].bar(default_by_age.index, default_by_age.values, alpha=0.7, color='coral')
axes[1,0].set_title('Default Rate by Age Group')
axes[1,0].set_ylabel('Default Rate')
axes[1,0].tick_params(axis='x', rotation=45)

# Credit purpose distribution
purpose_counts = X['purpose'].value_counts().head(8)
axes[1,1].bar(range(len(purpose_counts)), purpose_counts.values, alpha=0.7, color='purple')
axes[1,1].set_title('Top Credit Purposes')
axes[1,1].set_xticks(range(len(purpose_counts)))
axes[1,1].set_xticklabels(purpose_counts.index, rotation=45)

# Monthly income vs credit amount
axes[1,2].scatter(X['monthly_income'], X['credit_amount'], alpha=0.5, c=y, cmap='RdYlBu')
axes[1,2].set_title('Income vs Credit Amount')
axes[1,2].set_xlabel('Monthly Income ($)')
axes[1,2].set_ylabel('Credit Amount ($)')

plt.tight_layout()
plt.show()

## 2. Custom Feature Engineering Showcase

Now let's demonstrate the power of custom feature engineering by creating domain-specific financial features.

In [ ]:
# Initialize our custom classifier to explore feature engineering
print("🔧 Demonstrating Custom Feature Engineering...")

# Create classifier with all feature engineering enabled
model_full_features = CustomCreditRiskClassifier(
    enable_debt_ratio=True,
    enable_interaction_terms=True,
    enable_risk_scoring=True,
    decision_threshold=0.6
)

# Apply feature engineering to see what gets created
X_engineered = model_full_features._engineer_features(X)

print(f"📊 Original features: {X.shape[1]}")
print(f"🚀 Engineered features: {X_engineered.shape[1]}")
print(f"➕ Added {X_engineered.shape[1] - X.shape[1]} new features through custom engineering")

# Show the new features created
original_features = set(X.columns)
new_features = [col for col in X_engineered.columns if col not in original_features]

print("\n🎯 New features created:")
for i, feature in enumerate(new_features, 1):
    print(f"{i:2d}. {feature}")

In [ ]:
# Analyze the impact of feature engineering
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Impact of Custom Feature Engineering', fontsize=16)

# Debt-to-income ratio distribution
if 'debt_to_income_ratio' in X_engineered.columns:
    debt_ratios = X_engineered['debt_to_income_ratio']
    axes[0,0].hist([debt_ratios[y==0], debt_ratios[y==1]], 
                   bins=30, alpha=0.7, label=['Good Credit', 'Bad Credit'], color=['green', 'red'])
    axes[0,0].set_title('Debt-to-Income Ratio Distribution')
    axes[0,0].set_xlabel('Debt-to-Income Ratio')
    axes[0,0].legend()

# Age-credit interaction
if 'age_credit_interaction' in X_engineered.columns:
    age_credit_int = X_engineered['age_credit_interaction']
    axes[0,1].scatter(age_credit_int, y, alpha=0.5, c=y, cmap='RdYlBu')
    axes[0,1].set_title('Age-Credit Amount Interaction')
    axes[0,1].set_xlabel('Age × log(Credit Amount)')
    axes[0,1].set_ylabel('Default Risk')

# Stability score
if 'stability_score' in X_engineered.columns:
    stability = X_engineered['stability_score']
    axes[1,0].hist([stability[y==0], stability[y==1]], 
                   bins=20, alpha=0.7, label=['Good Credit', 'Bad Credit'], color=['green', 'red'])
    axes[1,0].set_title('Stability Score Distribution')
    axes[1,0].set_xlabel('Stability Score')
    axes[1,0].legend()

# Composite risk score
if 'composite_risk_score' in X_engineered.columns:
    risk_score = X_engineered['composite_risk_score']
    axes[1,1].hist([risk_score[y==0], risk_score[y==1]], 
                   bins=20, alpha=0.7, label=['Good Credit', 'Bad Credit'], color=['green', 'red'])
    axes[1,1].set_title('Composite Risk Score Distribution')
    axes[1,1].set_xlabel('Composite Risk Score')
    axes[1,1].legend()

plt.tight_layout()
plt.show()

# Show correlation of new features with target
correlations = []
for feature in new_features:
    if pd.api.types.is_numeric_dtype(X_engineered[feature]):
        corr = np.corrcoef(X_engineered[feature], y)[0,1]
        correlations.append((feature, abs(corr)))

correlations.sort(key=lambda x: x[1], reverse=True)

print("\n📈 Feature correlations with default risk (absolute values):")
for feature, corr in correlations[:10]:
    print(f"{feature:<35}: {corr:.3f}")

## 3. Model Training and Comparison

Let's train different model configurations and compare their performance.

In [ ]:
# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📚 Training set: {X_train.shape[0]} samples")
print(f"🧪 Test set: {X_test.shape[0]} samples")
print(f"📊 Training default rate: {y_train.mean():.1%}")
print(f"📊 Test default rate: {y_test.mean():.1%}")

In [ ]:
# Train different model configurations
print("🏋️ Training different model configurations...\n")

models = {
    'Baseline (No Custom Features)': CustomCreditRiskClassifier(
        enable_debt_ratio=False,
        enable_interaction_terms=False,
        enable_risk_scoring=False,
        decision_threshold=0.5
    ),
    'Debt Ratios Only': CustomCreditRiskClassifier(
        enable_debt_ratio=True,
        enable_interaction_terms=False,
        enable_risk_scoring=False,
        decision_threshold=0.5
    ),
    'Risk Scoring Only': CustomCreditRiskClassifier(
        enable_debt_ratio=False,
        enable_interaction_terms=False,
        enable_risk_scoring=True,
        decision_threshold=0.5
    ),
    'Full Feature Engineering': CustomCreditRiskClassifier(
        enable_debt_ratio=True,
        enable_interaction_terms=True,
        enable_risk_scoring=True,
        decision_threshold=0.5
    ),
    'Conservative (Full + Low Threshold)': CustomCreditRiskClassifier(
        enable_debt_ratio=True,
        enable_interaction_terms=True,
        enable_risk_scoring=True,
        decision_threshold=0.3  # More conservative
    )
}

# Train all models
trained_models = {}
for name, model in models.items():
    print(f"Training: {name}")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("\n✅ All models trained successfully!")

In [ ]:
# Evaluate model performance
print("📊 Model Performance Comparison\n")
print(f"{'Model':<35} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'AUC':<10}")
print("-" * 85)

results = {}
for name, model in trained_models.items():
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name:<35} {accuracy:<10.3f} {precision:<10.3f} {recall:<10.3f} {f1:<10.3f} {auc:<10.3f}")

# Find best performing model
best_model_name = max(results.keys(), key=lambda x: results[x]['auc'])
print(f"\n🏆 Best performing model: {best_model_name} (AUC: {results[best_model_name]['auc']:.3f})")

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC Curves
for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['probabilities'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={result['auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Performance metrics comparison
metrics_df = pd.DataFrame({
    name: [result['accuracy'], result['precision'], result['recall'], result['f1'], result['auc']]
    for name, result in results.items()
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'])

metrics_df.T.plot(kind='bar', ax=axes[1], alpha=0.8)
axes[1].set_title('Performance Metrics Comparison')
axes[1].set_ylabel('Score')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Feature Importance and Model Interpretability

In [ ]:
# Analyze feature importance for the best model
best_model = trained_models[best_model_name]
feature_importance = best_model.get_feature_importance()

if feature_importance is not None:
    # Get feature names after engineering
    X_engineered_sample = best_model._engineer_features(X_train.head(1))
    feature_names = X_engineered_sample.columns.tolist()
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    # Plot top 15 most important features
    plt.figure(figsize=(12, 8))
    top_features = importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['importance'], alpha=0.8)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance (|Coefficient|)')
    plt.title(f'Top 15 Most Important Features - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("🎯 Top 10 Most Important Features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
        print(f"{i:2d}. {row['feature']:<35}: {row['importance']:.4f}")
else:
    print("⚠️ Feature importance not available for this model")

## 5. Business Impact Analysis

In [ ]:
# Analyze business impact of model decisions
print("💼 Business Impact Analysis")
print("=" * 50)

# Use the best model for business analysis
y_pred_best = results[best_model_name]['predictions']
y_proba_best = results[best_model_name]['probabilities']

# Calculate confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_best).ravel()

print(f"📊 Confusion Matrix:")
print(f"   True Negatives (Correctly Approved):  {tn:3d}")
print(f"   False Positives (Incorrectly Rejected): {fp:3d}")
print(f"   False Negatives (Incorrectly Approved): {fn:3d}")
print(f"   True Positives (Correctly Rejected):   {tp:3d}")

# Calculate financial impact
# Assumptions: 
# - Average profit margin on good loans: 15%
# - Average loss on bad loans: 80%
# - Opportunity cost of rejected good loans: 5%

avg_credit_amount = X_test['credit_amount'].mean()

profit_per_good_loan = avg_credit_amount * 0.15
loss_per_bad_loan = avg_credit_amount * 0.80
opportunity_cost_per_rejection = avg_credit_amount * 0.05

# Calculate financial outcomes
profit_from_approved_good = tn * profit_per_good_loan
loss_from_approved_bad = fn * loss_per_bad_loan
opportunity_cost_from_rejected_good = fp * opportunity_cost_per_rejection
savings_from_rejected_bad = tp * loss_per_bad_loan

total_impact = profit_from_approved_good - loss_from_approved_bad - opportunity_cost_from_rejected_good

print(f"\n💰 Financial Impact Analysis:")
print(f"   Profit from approved good loans:     ${profit_from_approved_good:>12,.0f}")
print(f"   Loss from approved bad loans:       ${loss_from_approved_bad:>12,.0f}")
print(f"   Opportunity cost (rejected good):    ${opportunity_cost_from_rejected_good:>12,.0f}")
print(f"   Savings from rejected bad loans:     ${savings_from_rejected_bad:>12,.0f}")
print(f"   {'-'*50}")
print(f"   Net Financial Impact:               ${total_impact:>12,.0f}")

# Compare with a naive "approve all" strategy
total_good_loans = tn + fp
total_bad_loans = tp + fn
naive_profit = total_good_loans * profit_per_good_loan
naive_loss = total_bad_loans * loss_per_bad_loan
naive_total = naive_profit - naive_loss

improvement = total_impact - naive_total

print(f"\n📈 Comparison with 'Approve All' Strategy:")
print(f"   Naive approach net impact:          ${naive_total:>12,.0f}")
print(f"   Model-based approach:               ${total_impact:>12,.0f}")
print(f"   Improvement:                        ${improvement:>12,.0f}")
print(f"   Improvement percentage:             {improvement/abs(naive_total)*100:>11.1f}%")

In [ ]:
# Risk-based decision making
print("\n🎯 Risk-Based Decision Recommendations")
print("=" * 50)

# Create decision framework based on risk scores
decision_framework = []
for i, (risk_score, actual, predicted, amount) in enumerate(zip(y_proba_best, y_test, y_pred_best, X_test['credit_amount'])):
    if risk_score <= 0.2:
        decision = "APPROVE - Prime Rate"
        recommended_amount = amount
        interest_premium = 0.0
    elif risk_score <= 0.4:
        decision = "APPROVE - Standard Rate"
        recommended_amount = amount * 0.9
        interest_premium = 1.0
    elif risk_score <= 0.6:
        decision = "APPROVE - Higher Rate"
        recommended_amount = amount * 0.7
        interest_premium = 2.0
    elif risk_score <= 0.8:
        decision = "REVIEW - Manual Underwriting"
        recommended_amount = amount * 0.5
        interest_premium = 3.0
    else:
        decision = "REJECT - Too High Risk"
        recommended_amount = 0
        interest_premium = None
    
    decision_framework.append({
        'risk_score': risk_score,
        'decision': decision,
        'requested_amount': amount,
        'recommended_amount': recommended_amount,
        'interest_premium': interest_premium,
        'actual_outcome': actual,
        'predicted_outcome': predicted
    })

df_decisions = pd.DataFrame(decision_framework)

# Summary by decision category
decision_summary = df_decisions.groupby('decision').agg({
    'risk_score': ['count', 'mean'],
    'actual_outcome': 'mean',
    'recommended_amount': 'sum',
    'requested_amount': 'sum'
}).round(3)

decision_summary.columns = ['Count', 'Avg_Risk_Score', 'Actual_Default_Rate', 'Total_Recommended', 'Total_Requested']

print("Decision Category Summary:")
display(decision_summary)

# Show some specific examples
print("\n📋 Sample Decisions:")
sample_decisions = df_decisions.sample(10, random_state=42)[[
    'risk_score', 'decision', 'requested_amount', 'recommended_amount', 'actual_outcome'
]].round(3)
display(sample_decisions)

## 6. IntegratedML Integration Examples

Now let's see how this model would integrate with IntegratedML for database-resident processing.

In [ ]:
# Save the best model for IntegratedML integration
model_path = Path('../models/credit_risk_model.pkl')
model_path.parent.mkdir(exist_ok=True)

best_model.save_model(str(model_path))
print(f"💾 Model saved to: {model_path}")

# Show model configuration for IntegratedML
print("\n🔧 Model Configuration for IntegratedML:")
model_config = {
    'model_class': 'demos.credit_risk.models.credit_risk_classifier.CustomCreditRiskClassifier',
    'parameters': best_model.get_params(),
    'feature_engineering': {
        'debt_ratio': best_model.enable_debt_ratio,
        'interaction_terms': best_model.enable_interaction_terms,
        'risk_scoring': best_model.enable_risk_scoring
    },
    'performance': {
        'accuracy': results[best_model_name]['accuracy'],
        'auc': results[best_model_name]['auc'],
        'precision': results[best_model_name]['precision'],
        'recall': results[best_model_name]['recall']
    }
}

import json
print(json.dumps(model_config, indent=2))

In [ ]:
# Generate SQL examples for IntegratedML integration
print("📝 SQL Examples for IntegratedML Integration:")
print("=" * 50)

# Model creation SQL
sql_create_model = f"""
-- Create Credit Risk Model in IntegratedML
CREATE MODEL CreditRiskModel PREDICTING (default_risk)
FROM CreditApplications 
USING "demos.credit_risk.models.credit_risk_classifier.CustomCreditRiskClassifier"(
    enable_debt_ratio={str(best_model.enable_debt_ratio).lower()},
    enable_interaction_terms={str(best_model.enable_interaction_terms).lower()},
    enable_risk_scoring={str(best_model.enable_risk_scoring).lower()},
    decision_threshold={best_model.decision_threshold}
);
"""

print("1. Model Creation:")
print(sql_create_model)

# Prediction examples
sql_predictions = """
-- Make predictions on new applications
SELECT 
    application_id,
    customer_id,
    credit_amount,
    PREDICT(CreditRiskModel USING *) as risk_probability,
    PREDICT(CreditRiskModel WITH 'class' USING *) as risk_decision,
    CASE 
        WHEN PREDICT(CreditRiskModel USING *) <= 0.2 THEN 'APPROVE - Prime Rate'
        WHEN PREDICT(CreditRiskModel USING *) <= 0.4 THEN 'APPROVE - Standard Rate'
        WHEN PREDICT(CreditRiskModel USING *) <= 0.6 THEN 'APPROVE - Higher Rate'
        WHEN PREDICT(CreditRiskModel USING *) <= 0.8 THEN 'REVIEW - Manual Underwriting'
        ELSE 'REJECT - Too High Risk'
    END as business_decision
FROM NewCreditApplications
WHERE application_status = 'PENDING'
ORDER BY risk_probability DESC;
"""

print("2. Risk Assessment Predictions:")
print(sql_predictions)

# Performance monitoring
sql_monitoring = """
-- Monitor model performance over time
SELECT 
    DATE_TRUNC('week', application_date) as week_start,
    COUNT(*) as total_applications,
    AVG(default_risk) as actual_default_rate,
    AVG(PREDICT(CreditRiskModel USING *)) as predicted_default_rate,
    AVG(ABS(default_risk - PREDICT(CreditRiskModel USING *))) as mean_absolute_error
FROM CreditApplications
WHERE default_risk IS NOT NULL
    AND application_date >= DATEADD(month, -3, CURRENT_DATE)
GROUP BY DATE_TRUNC('week', application_date)
ORDER BY week_start DESC;
"""

print("3. Performance Monitoring:")
print(sql_monitoring)

## 7. Key Insights and Next Steps

In [ ]:
print("🎯 Key Insights from Credit Risk Assessment Demo")
print("=" * 55)

print("\n📊 Model Performance:")
print(f"   • Best model: {best_model_name}")
print(f"   • AUC Score: {results[best_model_name]['auc']:.3f}")
print(f"   • Accuracy: {results[best_model_name]['accuracy']:.1%}")
print(f"   • Precision: {results[best_model_name]['precision']:.1%}")
print(f"   • Recall: {results[best_model_name]['recall']:.1%}")

print("\n🚀 Feature Engineering Impact:")
baseline_auc = results['Baseline (No Custom Features)']['auc']
best_auc = results[best_model_name]['auc']
improvement = (best_auc - baseline_auc) / baseline_auc * 100
print(f"   • Baseline AUC: {baseline_auc:.3f}")
print(f"   • Best AUC: {best_auc:.3f}")
print(f"   • Improvement: {improvement:.1f}%")
print(f"   • Added features: {X_engineered.shape[1] - X.shape[1]}")

print("\n💰 Business Value:")
print(f"   • Net financial impact: ${total_impact:,.0f}")
print(f"   • Improvement over naive approach: ${improvement:,.0f}")
print(f"   • Model correctly identifies {(tp + tn)/(tp + tn + fp + fn):.1%} of cases")

print("\n🔒 Security & Compliance Benefits:")
print("   • Sensitive financial data never leaves the database")
print("   • Custom feature engineering runs in secure environment")
print("   • Full audit trail of model decisions")
print("   • Meets financial regulatory requirements")

print("\n⚡ Performance Benefits:")
print("   • No data movement reduces latency")
print("   • Real-time scoring capability")
print("   • Scales with database infrastructure")
print("   • Consistent preprocessing pipeline")

print("\n🎓 Learning Outcomes:")
print("   ✅ Implemented custom feature engineering in IntegratedML context")
print("   ✅ Demonstrated domain-specific financial transformations")
print("   ✅ Showed business impact of model decisions")
print("   ✅ Integrated with familiar scikit-learn patterns")
print("   ✅ Maintained enterprise security and compliance")

print("\n🚀 Next Steps:")
print("   1. Deploy model to IntegratedML environment")
print("   2. Set up real-time scoring pipeline")
print("   3. Implement performance monitoring dashboard")
print("   4. Train business users on interpretation")
print("   5. Explore more advanced feature engineering")

print("\n📖 Continue Learning:")
print("   • Next Demo: Fraud Detection with Ensemble Models")
print("   • Advanced Demo: Sales Forecasting with Third-party Libraries")
print("   • Documentation: IntegratedML Architecture Deep Dive")

## Conclusion

This demo showcased how **IntegratedML's pluggable models capability** enables sophisticated, domain-specific machine learning directly within database workflows. Key achievements:

### 🎯 **Technical Success**
- **Custom Feature Engineering**: Created meaningful financial indicators that improved model performance
- **Multiple Model Configurations**: Demonstrated flexibility in business requirements
- **Performance Improvement**: Achieved significant gains over baseline approaches

### 💼 **Business Value**
- **Risk-Based Decisions**: Automated credit approval with nuanced risk assessment
- **Financial Impact**: Quantified business value of model-driven decisions
- **Operational Efficiency**: Eliminated data movement and reduced processing time

### 🔒 **Enterprise Benefits**
- **Data Security**: Sensitive financial data never leaves the secure database environment
- **Regulatory Compliance**: Maintained audit trails and data governance requirements
- **Scalability**: Leveraged database infrastructure for consistent performance

### 🛠️ **Developer Experience**
- **Familiar Patterns**: Used standard scikit-learn interfaces with enterprise enhancements
- **Easy Integration**: Simple SQL commands for model deployment and scoring
- **Comprehensive Tooling**: Full lifecycle support from development to production

---

**Ready for the next challenge?** Check out the [Fraud Detection Ensemble Demo](../fraud_detection/README.md) to learn about real-time multi-model orchestration!